# SDTM CT Term NCIt Synonyms — NCIt synonyms for every SDTM CT term

Writes `sdtm-test-codes/interim/SDTM_CT_Term_NCIt_Synonyms.csv`: one row per NCIt code of a term
in SDTM CT (every codelist, not only test codes), with the NCIt synonyms from the NCI EVS FLAT file.

The four Findings consumers in `sdtm-findings-graph/` read it for the `term_ncit_synonyms` column
of their `Codelists` sheet. SDTM CT itself carries CDISC synonyms only; NCIt synonyms are NCI's,
and the FLAT file does not say where each synonym comes from.

Also writes `sdtm-test-codes/interim/SDTM_CT_Term_NCIt_Parents.csv`: for the same term codes, the
NCIt parents from the FLAT `parents` column (C-codes, and their NCIt preferred names in the same
order, each joined with `|`). NCI's hierarchy, not CDISC's. Immunogenicity_Findings reads it for
`NCIt_Parent_Codes` / `NCIt_Parent_Terms` in its Test_Identity sheet.

Also writes `sdtm-test-codes/interim/SDTM_CT_NCIt_Ancestor_Graph.csv`: one row per NCIt concept that
is an SDTM CT term or an NCIt ancestor of one, with its preferred name, its NCIt synonyms (the rule
below), its parents (C-codes joined with `|`) and `In_SDTM_CT` (Yes / No). The NCIt hierarchy above
the CT terms, NCI's, not CDISC's; a consumer computes the ancestor chain of a term from it.
Immunogenicity_Findings reads it for its Test_Ancestors sheet.

## Rule

The same rule as `NCIt_Synonyms` in `SDTM_CT_NCIt_Enrich` (SDTM_Test_Identity): the first FLAT
entry is the preferred name and is dropped; the rest are deduplicated ignoring case, in FLAT order,
and joined with `|`, the FLAT file's own separator (as `NCIt_Synonyms` in SDTM_Test_Identity since
2026-10-30). Some NCIt synonyms contain `;` (e.g. *Esophageal; Distal*); `|` never occurs inside one.

## Version

The FLAT file comes from a URL without a version and carries no release number. The pin
`NCIt` in `Reference_Versions.json` is the build date of `Thesaurus.txt` inside
`Thesaurus.FLAT.zip`; the run **fails** when they differ. The date is written to every row
(`NCIt_FLAT_Date`) so the consumers can check it too.

## Inputs

| File | Use |
|---|---|
| `downloads/SDTM_Terminology.txt` | the term codes (rows with a Codelist Code) |
| `downloads/Thesaurus.FLAT.zip`, `downloads/Thesaurus.txt` | NCIt synonyms; the build date |

Both are downloaded by `SDTM_CT_NCIt_Enrich`. Run this notebook after it.

The run **fails** when a term code is not in the FLAT file.


## 1. Setup

In [ ]:
import json
import zipfile
from datetime import date
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent          # sdtm-test-codes/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))

SDTM_FILE = BASE_DIR / 'downloads' / 'SDTM_Terminology.txt'
FLAT_ZIP = BASE_DIR / 'downloads' / 'Thesaurus.FLAT.zip'
FLAT_FILE = BASE_DIR / 'downloads' / 'Thesaurus.txt'
OUTPUT_FILE = BASE_DIR / 'interim' / 'SDTM_CT_Term_NCIt_Synonyms.csv'
PARENTS_FILE = BASE_DIR / 'interim' / 'SDTM_CT_Term_NCIt_Parents.csv'
GRAPH_FILE = BASE_DIR / 'interim' / 'SDTM_CT_NCIt_Ancestor_Graph.csv'

for f in [SDTM_FILE, FLAT_ZIP, FLAT_FILE]:
    if not f.exists():
        raise FileNotFoundError(f'{f} not found: run SDTM_CT_NCIt_Enrich first')
    print(f'  {f.relative_to(REPO_ROOT)}')


## 2. Check the NCIt pin against the FLAT build date

In [ ]:
with zipfile.ZipFile(FLAT_ZIP) as zf:
    _info = zf.getinfo('Thesaurus.txt')
FLAT_DATE = date(*_info.date_time[:3]).isoformat()
if _info.file_size != FLAT_FILE.stat().st_size:
    raise RuntimeError(f'{FLAT_FILE.name} ({FLAT_FILE.stat().st_size:,} bytes) is not the file in '
                       f'{FLAT_ZIP.name} ({_info.file_size:,} bytes): unzip again')
if FLAT_DATE != VERSIONS['NCIt']:
    raise RuntimeError(f"NCIt pin {VERSIONS['NCIt']} != FLAT build date {FLAT_DATE}: update Reference_Versions.json")
print(f'NCIt FLAT build date: {FLAT_DATE} (= pin)')


## 3. Term codes and NCIt synonyms

In [ ]:
ct = pd.read_csv(SDTM_FILE, sep='\t', dtype=str).fillna('')
term_codes = sorted(set(ct.loc[ct['Codelist Code'] != '', 'Code']))
print(f'SDTM CT term codes: {len(term_codes):,}')

FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']
flat = pd.read_csv(FLAT_FILE, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
# NCIt preferred name of every concept (first FLAT synonym), for the parent names below.
pref_name = {c: v.split('|')[0].strip() for c, v in zip(flat['code'], flat['synonyms'])}
flat = flat[flat['code'].isin(term_codes)]
missing = sorted(set(term_codes) - set(flat['code']))
if missing:
    raise RuntimeError(f'{len(missing)} SDTM CT term codes not in the FLAT file, e.g. {missing[:10]}')


def ncit_synonyms(value):
    entries = [s.strip() for s in value.split('|') if s.strip()][1:]   # drop the preferred name
    seen, kept = set(), []
    for s in entries:
        if s.lower() not in seen:
            seen.add(s.lower())
            kept.append(s)
    return kept


rows = []
semicolon = 0
for code, value in zip(flat['code'], flat['synonyms']):
    syns = ncit_synonyms(value)
    semicolon += sum(';' in s for s in syns)
    rows.append({'NCIt_Code': code, 'NCIt_Synonyms': '|'.join(syns), 'NCIt_FLAT_Date': FLAT_DATE})
out = pd.DataFrame(rows).sort_values('NCIt_Code').reset_index(drop=True)
print(f'Rows: {len(out):,}; with NCIt synonyms: {(out["NCIt_Synonyms"] != "").sum():,}')
print(f'NCIt synonyms that contain ";" (kept whole by the "|" separator): {semicolon}')

# NCIt parents of each term code, from the FLAT parents column, with their NCIt preferred names.
parent_rows = []
for code, value in zip(flat['code'], flat['parents']):
    parents = [p for p in value.split('|') if p]
    no_name = [p for p in parents if not pref_name.get(p)]
    if no_name:
        raise RuntimeError(f'{code}: parent codes without a FLAT row: {no_name}')
    parent_rows.append({'NCIt_Code': code, 'NCIt_Parent_Codes': '|'.join(parents),
                        'NCIt_Parent_Terms': '|'.join(pref_name[p] for p in parents), 'NCIt_FLAT_Date': FLAT_DATE})
parents_out = pd.DataFrame(parent_rows).sort_values('NCIt_Code').reset_index(drop=True)
print(f'Parents: {len(parents_out):,} rows; with parents: {(parents_out["NCIt_Parent_Codes"] != "").sum():,}')


Rows: 26,965; with NCIt synonyms: 26,945
NCIt synonyms that contain ";" (kept whole by the "|" separator): 81


## 3b. The NCIt hierarchy above the CT terms

In [ ]:
# Every term code and every NCIt ancestor of one, from the FLAT parents column: one row per concept.
# The FLAT file is read again in full here; the cell above keeps only the term codes.
flat_all = pd.read_csv(FLAT_FILE, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
parents_of = {c: [p for p in v.split('|') if p] for c, v in zip(flat_all['code'], flat_all['parents'])}
synonyms_of = dict(zip(flat_all['code'], flat_all['synonyms']))

graph_codes = set(term_codes)
todo = list(term_codes)
while todo:
    code = todo.pop()
    for p in parents_of[code]:
        if p not in parents_of:
            raise RuntimeError(f'{code}: parent {p} has no FLAT row')
        if p not in graph_codes:
            graph_codes.add(p)
            todo.append(p)

graph_out = pd.DataFrame({'NCIt_Code': sorted(graph_codes)})
graph_out['NCIt_Preferred_Term'] = graph_out['NCIt_Code'].map(pref_name)
graph_out['NCIt_Synonyms'] = graph_out['NCIt_Code'].map(lambda c: '|'.join(ncit_synonyms(synonyms_of[c])))
graph_out['NCIt_Parent_Codes'] = graph_out['NCIt_Code'].map(lambda c: '|'.join(parents_of[c]))
graph_out['In_SDTM_CT'] = graph_out['NCIt_Code'].isin(set(term_codes)).map({True: 'Yes', False: 'No'})
graph_out['NCIt_FLAT_Date'] = FLAT_DATE
if (graph_out['NCIt_Preferred_Term'] == '').any():
    raise RuntimeError('Concepts without an NCIt preferred name in the FLAT file')

# The term rows say what the two files above say (same FLAT file, same rule).
_terms = graph_out[graph_out['In_SDTM_CT'] == 'Yes'].set_index('NCIt_Code')
if _terms.loc[parents_out['NCIt_Code'], 'NCIt_Parent_Codes'].tolist() != parents_out['NCIt_Parent_Codes'].tolist():
    raise RuntimeError('Ancestor graph and SDTM_CT_Term_NCIt_Parents disagree on parents')
if _terms.loc[out['NCIt_Code'], 'NCIt_Synonyms'].tolist() != out['NCIt_Synonyms'].tolist():
    raise RuntimeError('Ancestor graph and SDTM_CT_Term_NCIt_Synonyms disagree on synonyms')
print(f'Ancestor graph: {len(graph_out):,} concepts ({(graph_out["In_SDTM_CT"] == "Yes").sum():,} SDTM CT terms, '
      f'{(graph_out["In_SDTM_CT"] == "No").sum():,} NCIt ancestors only)')

## 4. Write

In [ ]:
out.to_csv(OUTPUT_FILE, index=False)
print(f'Written: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
parents_out.to_csv(PARENTS_FILE, index=False)
print(f'Written: {PARENTS_FILE.relative_to(REPO_ROOT)}')
graph_out.to_csv(GRAPH_FILE, index=False)
print(f'Written: {GRAPH_FILE.relative_to(REPO_ROOT)}')
